In [1]:
from google.colab import files
uploaded = files.upload()

Saving Sample - Superstore.csv to Sample - Superstore.csv


In [2]:
import pandas as pd
import sqlite3


df = pd.read_csv('superstore.csv', encoding='latin1')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')

print(df.columns.tolist())
print(df.shape)
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'superstore.csv'

In [3]:
from google.colab import files
uploaded = files.upload()

Saving Sample - Superstore.csv to Sample - Superstore (1).csv


In [4]:
import os
print(os.listdir())

['.config', 'Sample - Superstore.csv', 'Sample - Superstore (1).csv', 'sample_data']


In [5]:
df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')


In [6]:
import pandas as pd
import sqlite3

df = pd.read_csv('superstore.csv', encoding='latin1')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')

print(df.columns.tolist())
print(df.shape)
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'superstore.csv'

In [7]:

import pandas as pd
import sqlite3

df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(3)

Shape: (9994, 21)
Columns: ['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


In [8]:
conn = sqlite3.connect('superstore.db')

# Load full data into superstore_raw
df.to_sql('superstore_raw', conn, if_exists='replace', index=False)
print("superstore_raw loaded:", pd.read_sql("SELECT COUNT(*) as total FROM superstore_raw", conn).values)

superstore_raw loaded: [[9994]]


In [9]:
cursor = conn.cursor()

cursor.executescript("""
    DROP TABLE IF EXISTS customers;
    DROP TABLE IF EXISTS products;
    DROP TABLE IF EXISTS orders;

    CREATE TABLE customers AS
    SELECT DISTINCT
        customer_id,
        customer_name,
        segment,
        country,
        city,
        state,
        region
    FROM superstore_raw;

    CREATE TABLE products AS
    SELECT DISTINCT
        product_id,
        product_name,
        category,
        sub_category
    FROM superstore_raw;

    CREATE TABLE orders AS
    SELECT
        row_id,
        order_id,
        order_date,
        ship_date,
        ship_mode,
        customer_id,
        product_id,
        CAST(sales AS REAL) AS sales,
        CAST(quantity AS INTEGER) AS quantity,
        CAST(discount AS REAL) AS discount,
        CAST(profit AS REAL) AS profit
    FROM superstore_raw;
""")

conn.commit()
print("Tables created.")

Tables created.


In [10]:
for table in ['customers', 'products', 'orders']:
    count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table}", conn)
    print(f"{table}: {count['cnt'][0]} rows")

customers: 4688 rows
products: 1894 rows
orders: 9994 rows


In [11]:
q1 = """
SELECT order_id, customer_id, sales
FROM orders
WHERE sales > (SELECT AVG(sales) FROM orders)
ORDER BY sales DESC
LIMIT 10;
"""
df_q1 = pd.read_sql(q1, conn)
print("Above Average Sales Orders:")
df_q1

Above Average Sales Orders:


,order_id,customer_id,sales
0,CA-2014-145317,SM-20320,22638.480
1,CA-2016-118689,TC-20980,17499.950
2,CA-2017-140151,RB-19360,13999.960
3,CA-2017-127180,TA-21385,11199.968
4,CA-2017-166709,HL-15040,10499.970
5,CA-2016-117121,AB-10105,9892.740
6,CA-2014-116904,SC-20095,9449.950
7,US-2016-107440,BS-11365,9099.930
8,CA-2016-158841,SE-20110,8749.950
9,CA-2016-143714,CC-12370,8399.976


In [12]:
q2 = """
SELECT o.customer_id, o.order_id, o.sales
FROM orders o
WHERE o.sales = (
    SELECT MAX(o2.sales)
    FROM orders o2
    WHERE o2.customer_id = o.customer_id
)
ORDER BY o.sales DESC
LIMIT 10;
"""
df_q2 = pd.read_sql(q2, conn)
print("Highest Order per Customer (Top 10):")
df_q2

Highest Order per Customer (Top 10):


,customer_id,order_id,sales
0,SM-20320,CA-2014-145317,22638.480
1,TC-20980,CA-2016-118689,17499.950
2,RB-19360,CA-2017-140151,13999.960
3,TA-21385,CA-2017-127180,11199.968
4,HL-15040,CA-2017-166709,10499.970
5,AB-10105,CA-2016-117121,9892.740
6,SC-20095,CA-2014-116904,9449.950
7,BS-11365,US-2016-107440,9099.930
8,SE-20110,CA-2016-158841,8749.950
9,CC-12370,CA-2016-143714,8399.976


In [13]:
q3 = """
WITH customer_sales AS (
    SELECT
        customer_id,
        ROUND(SUM(sales), 2) AS total_sales,
        COUNT(DISTINCT order_id) AS total_orders
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name,
    c.segment,
    c.region,
    cs.total_sales,
    cs.total_orders
FROM customer_sales cs
JOIN customers c ON cs.customer_id = c.customer_id
ORDER BY cs.total_sales DESC
LIMIT 10;
"""
df_q3 = pd.read_sql(q3, conn)
print("Top 10 Customers by Total Sales:")
df_q3

Top 10 Customers by Total Sales:


,customer_name,segment,region,total_sales,total_orders
0,Sean Miller,Home Office,South,25043.05,5
1,Sean Miller,Home Office,Central,25043.05,5
2,Sean Miller,Home Office,South,25043.05,5
3,Sean Miller,Home Office,West,25043.05,5
4,Sean Miller,Home Office,East,25043.05,5
5,Tamara Chand,Corporate,West,19052.22,5
6,Tamara Chand,Corporate,Central,19052.22,5
7,Tamara Chand,Corporate,Central,19052.22,5
8,Tamara Chand,Corporate,East,19052.22,5
9,Tamara Chand,Corporate,South,19052.22,5


In [14]:
q4 = """
WITH customer_sales AS (
    SELECT
        customer_id,
        ROUND(SUM(sales), 2) AS total_sales
    FROM orders
    GROUP BY customer_id
),
ranked AS (
    SELECT
        customer_id,
        total_sales,
        RANK() OVER (ORDER BY total_sales DESC) AS sales_rank,
        ROW_NUMBER() OVER (ORDER BY total_sales DESC) AS row_num
    FROM customer_sales
)
SELECT
    c.customer_name,
    c.segment,
    r.total_sales,
    r.sales_rank,
    r.row_num
FROM ranked r
JOIN customers c ON r.customer_id = c.customer_id
LIMIT 15;
"""
df_q4 = pd.read_sql(q4, conn)
print("Customer Sales Ranking (RANK + ROW_NUMBER):")
df_q4

Customer Sales Ranking (RANK + ROW_NUMBER):


,customer_name,segment,total_sales,sales_rank,row_num
0,Claire Gute,Consumer,1148.78,594,594
1,Darrin Van Huff,Corporate,1119.48,599,599
2,Sean O'Donnell,Consumer,2602.58,337,337
3,Brosina Hoffman,Consumer,6255.35,72,72
4,Andrew Allen,Consumer,1790.51,471,471
5,Irene Maddox,Consumer,4930.47,120,120
6,Harold Pawlan,Home Office,1990.31,441,441
7,Pete Kriz,Consumer,8646.93,30,30
8,Alejandro Grove,Consumer,2582.90,340,340
9,Zuschuss Donatelli,Consumer,1493.94,520,520


In [15]:
q5 = """
WITH customer_sales AS (
    SELECT customer_id, ROUND(SUM(sales), 2) AS total_sales
    FROM orders GROUP BY customer_id
)
SELECT c.customer_name, c.region, cs.total_sales,
       RANK() OVER (ORDER BY cs.total_sales DESC) AS rank
FROM customer_sales cs
JOIN customers c ON cs.customer_id = c.customer_id
LIMIT 5;
"""
df_q5 = pd.read_sql(q5, conn)
print("Top 5 Customers:")
df_q5

Top 5 Customers:


,customer_name,region,total_sales,rank
0,Sean Miller,South,25043.05,1
1,Sean Miller,Central,25043.05,1
2,Sean Miller,South,25043.05,1
3,Sean Miller,West,25043.05,1
4,Sean Miller,East,25043.05,1


In [16]:
q6 = """
WITH customer_sales AS (
    SELECT customer_id, ROUND(SUM(sales), 2) AS total_sales
    FROM orders GROUP BY customer_id
)
SELECT c.customer_name, c.region, cs.total_sales,
       RANK() OVER (ORDER BY cs.total_sales ASC) AS rank
FROM customer_sales cs
JOIN customers c ON cs.customer_id = c.customer_id
LIMIT 5;
"""
df_q6 = pd.read_sql(q6, conn)
print("Bottom 5 Customers (Lowest Sales):")
df_q6

Bottom 5 Customers (Lowest Sales):


,customer_name,region,total_sales,rank
0,Thais Sissman,West,4.83,1
1,Thais Sissman,South,4.83,1
2,Lela Donovan,Central,5.30,3
3,Carl Jackson,East,16.52,4
4,Mitch Gastineau,South,16.74,5


In [17]:
q7 = """
SELECT c.customer_name, c.segment, COUNT(DISTINCT o.order_id) AS order_count
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY o.customer_id
HAVING order_count = 1
ORDER BY c.customer_name
LIMIT 10;
"""
df_q7 = pd.read_sql(q7, conn)
print(f"Single-order customers (showing 10 of {len(pd.read_sql(q7, conn))}):")
df_q7

Single-order customers (showing 10 of 10):


,customer_name,segment,order_count
0,Anemone Ratner,Consumer,1
1,Anthony O'Donnell,Corporate,1
2,Carl Jackson,Corporate,1
3,Jenna Caffey,Consumer,1
4,Jocasta Rupert,Consumer,1
5,Lela Donovan,Corporate,1
6,Mitch Gastineau,Corporate,1
7,Patricia Hirasaki,Home Office,1
8,Ricardo Emerson,Consumer,1
9,Roland Murray,Consumer,1


In [18]:
q8 = """
WITH avg_sales AS (
    SELECT AVG(sales) AS avg_val FROM orders
),
high_orders AS (
    SELECT o.order_id, o.customer_id, o.sales, o.profit
    FROM orders o, avg_sales a
    WHERE o.sales > a.avg_val
)
SELECT
    c.customer_name,
    c.segment,
    h.order_id,
    ROUND(h.sales, 2) AS sales,
    ROUND(h.profit, 2) AS profit
FROM high_orders h
JOIN customers c ON h.customer_id = c.customer_id
ORDER BY h.sales DESC
LIMIT 10;
"""
df_q8 = pd.read_sql(q8, conn)
print("Above-Average Sales with Customer Details:")
df_q8

Above-Average Sales with Customer Details:


,customer_name,segment,order_id,sales,profit
0,Sean Miller,Home Office,CA-2014-145317,22638.48,-1811.08
1,Sean Miller,Home Office,CA-2014-145317,22638.48,-1811.08
2,Sean Miller,Home Office,CA-2014-145317,22638.48,-1811.08
3,Sean Miller,Home Office,CA-2014-145317,22638.48,-1811.08
4,Sean Miller,Home Office,CA-2014-145317,22638.48,-1811.08
5,Tamara Chand,Corporate,CA-2016-118689,17499.95,8399.98
6,Tamara Chand,Corporate,CA-2016-118689,17499.95,8399.98
7,Tamara Chand,Corporate,CA-2016-118689,17499.95,8399.98
8,Tamara Chand,Corporate,CA-2016-118689,17499.95,8399.98
9,Tamara Chand,Corporate,CA-2016-118689,17499.95,8399.98


In [19]:
## Key Insights

- Top customer drives significantly more revenue than average — classic Pareto pattern (few customers = most revenue).
- Several customers placed only 1 order — potential churn risk or one-time buyers.
- Above-average orders are concentrated in specific segments (likely Corporate/Home Office).
- RANK vs ROW_NUMBER difference: RANK gives same rank to ties; ROW_NUMBER always unique — useful for different business scenarios.
- Window functions avoid self-joins and make ranking logic cleaner and faster.

SyntaxError: invalid character '—' (U+2014) (1372376230.py, line 3)

In [20]:
 File "/tmp/ipykernel_3895/1372376230.py", line 3
    - Top customer drives significantly more revenue than average — classic Pareto pattern (few customers = most revenue).
                                                                  ^
SyntaxError: invalid character '—' (U+2014)


SyntaxError: invalid character '—' (U+2014) (2353105728.py, line 2)

## Key Insights

- Top customer drives significantly more revenue than average — classic Pareto pattern (few customers = most revenue).
- Several customers placed only 1 order — potential churn risk or one-time buyers.
- Above-average orders are concentrated in specific segments (likely Corporate/Home Office).
- RANK vs ROW_NUMBER difference: RANK gives same rank to ties; ROW_NUMBER always unique — useful for different business scenarios.
- Window functions avoid self-joins and make ranking logic cleaner and faster.

In [21]:
q_cte_subquery = """
WITH customer_sales AS (
    SELECT customer_id, ROUND(SUM(sales), 2) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name, c.segment, c.region, cs.total_sales
FROM customer_sales cs
JOIN customers c ON cs.customer_id = c.customer_id
WHERE cs.total_sales > (SELECT AVG(total_sales) FROM customer_sales)
ORDER BY cs.total_sales DESC;
"""
df_a = pd.read_sql(q_cte_subquery, conn)
print("Customers with Above-Average Total Sales:")
df_a

Customers with Above-Average Total Sales:


,customer_name,segment,region,total_sales
0,Sean Miller,Home Office,South,25043.05
1,Sean Miller,Home Office,Central,25043.05
2,Sean Miller,Home Office,South,25043.05
3,Sean Miller,Home Office,West,25043.05
4,Sean Miller,Home Office,East,25043.05
...,...,...,...,...
2106,Craig Yedwab,Corporate,South,2900.03
2107,Craig Yedwab,Corporate,West,2900.03
2108,Craig Yedwab,Corporate,Central,2900.03
2109,Craig Yedwab,Corporate,West,2900.03


In [22]:
q_rownumber = """
SELECT
    c.customer_name,
    o.order_id,
    o.order_date,
    ROUND(o.sales, 2) AS sales,
    ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.sales DESC) AS order_rank_within_customer
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
ORDER BY c.customer_name, order_rank_within_customer
LIMIT 20;
"""
df_b = pd.read_sql(q_rownumber, conn)
print("Row Numbers per Order within Each Customer:")
df_b

Row Numbers per Order within Each Customer:


,customer_name,order_id,order_date,sales,order_rank_within_customer
0,Aaron Bergman,CA-2016-140935,11/10/2016,341.96,1
1,Aaron Bergman,CA-2016-140935,11/10/2016,341.96,2
2,Aaron Bergman,CA-2016-140935,11/10/2016,341.96,3
3,Aaron Bergman,CA-2014-156587,3/7/2014,242.94,4
4,Aaron Bergman,CA-2014-156587,3/7/2014,242.94,5
5,Aaron Bergman,CA-2014-156587,3/7/2014,242.94,6
6,Aaron Bergman,CA-2016-140935,11/10/2016,221.98,7
7,Aaron Bergman,CA-2016-140935,11/10/2016,221.98,8
8,Aaron Bergman,CA-2016-140935,11/10/2016,221.98,9
9,Aaron Bergman,CA-2014-156587,3/7/2014,48.71,10


In [23]:
q_top3 = """
WITH customer_sales AS (
    SELECT customer_id, ROUND(SUM(sales), 2) AS total_sales
    FROM orders
    GROUP BY customer_id
),
ranked AS (
    SELECT
        customer_id,
        total_sales,
        RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
    FROM customer_sales
)
SELECT c.customer_name, c.segment, c.region, r.total_sales, r.sales_rank
FROM ranked r
JOIN customers c ON r.customer_id = c.customer_id
WHERE r.sales_rank <= 3;
"""
df_c = pd.read_sql(q_top3, conn)
print("Top 3 Customers by Total Sales:")
df_c

Top 3 Customers by Total Sales:


,customer_name,segment,region,total_sales,sales_rank
0,Raymond Buch,Consumer,East,15117.34,3
1,Tamara Chand,Corporate,West,19052.22,2
2,Sean Miller,Home Office,South,25043.05,1
3,Sean Miller,Home Office,Central,25043.05,1
4,Sean Miller,Home Office,South,25043.05,1
5,Raymond Buch,Consumer,Central,15117.34,3
6,Raymond Buch,Consumer,East,15117.34,3
7,Tamara Chand,Corporate,Central,19052.22,2
8,Raymond Buch,Consumer,South,15117.34,3
9,Tamara Chand,Corporate,Central,19052.22,2


In [24]:
q_final = """
WITH customer_sales AS (
    SELECT customer_id, ROUND(SUM(sales), 2) AS total_sales
    FROM orders
    GROUP BY customer_id
),
ranked_customers AS (
    SELECT
        customer_id,
        total_sales,
        RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
    FROM customer_sales
)
SELECT
    c.customer_name,
    c.segment,
    c.region,
    r.total_sales,
    r.sales_rank
FROM ranked_customers r
JOIN customers c ON r.customer_id = c.customer_id
ORDER BY r.sales_rank;
"""
df_final = pd.read_sql(q_final, conn)
print("Final Combined Query — Customer Name + Total Sales + Rank:")
df_final

Final Combined Query — Customer Name + Total Sales + Rank:


,customer_name,segment,region,total_sales,sales_rank
0,Sean Miller,Home Office,South,25043.05,1
1,Sean Miller,Home Office,Central,25043.05,1
2,Sean Miller,Home Office,South,25043.05,1
3,Sean Miller,Home Office,West,25043.05,1
4,Sean Miller,Home Office,East,25043.05,1
...,...,...,...,...,...
4683,Mitch Gastineau,Corporate,South,16.74,790
4684,Carl Jackson,Corporate,East,16.52,791
4685,Lela Donovan,Corporate,Central,5.30,792
4686,Thais Sissman,Consumer,West,4.83,793
